In [14]:
# ================================================================
#   FULL CREDIT-RISK XAI PIPELINE
#   LGBM + OPTUNA + SHAP + LIME
#   Works perfectly with Kaggle's "Credit Risk Dataset"
# ================================================================

import pandas as pd
import numpy as np
import shap
import optuna
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

# ================================================================
# 1. LOAD DATA
# ================================================================
DATA_PATH = "C:\\Users\\Admin\\Downloads\\credit_risk_dataset.csv"
TARGET = "loan_status"   # For Kaggle credit dataset

df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])

print("Dataset shape:", df.shape)
print("Preview:\n", df.head())

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()


print("\nCategorical columns:", cat_cols)
print("Numerical columns:", num_cols)

# ================================================================
# 2. PREPROCESSING PIPELINE
# ================================================================
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", SimpleImputer(strategy="median"), num_cols)
])


# ================================================================
# 3. OPTUNA HYPERPARAMETER TUNING
# ================================================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", -1, 15),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }

    model = LGBMClassifier(**params, random_state=42)

    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)
    return auc


print("\n🔍 Running Optuna hyperparameter tuning... (20 trials)")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
best_params = study.best_params

print("\n🎯 Best parameters:", best_params)

# ================================================================
# 4. TRAIN FINAL MODEL
# ================================================================
final_model = LGBMClassifier(**best_params, random_state=42)

pipeline = Pipeline([
    ("prep", preprocess),
    ("model", final_model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

pipeline.fit(X_train, y_train)

# ================================================================
# 5. MODEL EVALUATION
# ================================================================
pred = pipeline.predict(X_test)
prob = pipeline.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===\n")
print(classification_report(y_test, pred))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, pred))

print("\nROC-AUC Score:", roc_auc_score(y_test, prob))


# ================================================================
# 6. SHAP EXPLANATIONS
# ================================================================
print("\n📌 Computing SHAP values...")

preprocessed_X_test = pipeline.named_steps["prep"].transform(X_test)
model_only = pipeline.named_steps["model"]

explainer = shap.TreeExplainer(model_only)
shap_values = explainer.shap_values(preprocessed_X_test)

os.makedirs("xai_outputs", exist_ok=True)

print("📊 Generating SHAP summary plot...")
plt.figure()
shap.summary_plot(shap_values, preprocessed_X_test, show=False)
plt.savefig("xai_outputs/shap_summary.png", dpi=300, bbox_inches="tight")
plt.close()

# Top feature dependence plot
importances = model_only.feature_importances_
top_idx = np.argsort(importances)[-1]

plt.figure()
shap.dependence_plot(top_idx, shap_values, preprocessed_X_test, show=False)
plt.savefig("xai_outputs/shap_dependence.png", dpi=300, bbox_inches="tight")
plt.close()

print("SHAP plots saved in xai_outputs/")

# ================================================================
# 7. LIME LOCAL EXPLANATIONS
# ================================================================
print("\n📌 Running LIME explanations...")

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(preprocessed_X_test),
    feature_names=[f"f{i}" for i in range(preprocessed_X_test.shape[1])],
    mode="classification"
)

# Choose 3 random customers to explain
sample_indices = np.random.choice(len(X_test), 3, replace=False)

for idx in sample_indices:
    exp = lime_explainer.explain_instance(
        preprocessed_X_test[idx],
        model_only.predict_proba,
        num_features=10
    )

    exp.save_to_file(f"xai_outputs/lime_explanation_{idx}.html")
    print(f"✔ LIME explanation saved: lime_explanation_{idx}.html")


# ================================================================
# 8. SAVE MODEL + PIPELINE
# ================================================================
joblib.dump(pipeline, "xai_outputs/credit_risk_model.pkl")
print("\n📦 Model saved to xai_outputs/credit_risk_model.pkl")


print("\n🎉 FULL XAI PIPELINE COMPLETED SUCCESSFULLY!")


[I 2025-11-21 10:19:03,738] A new study created in memory with name: no-name-82e99433-c33d-4809-ae13-cf5408529222


Dataset shape: (32416, 12)
Preview:
    person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                    

[I 2025-11-21 10:19:07,479] Trial 0 finished with value: 0.9477336395127337 and parameters: {'n_estimators': 551, 'learning_rate': 0.07224871487620538, 'max_depth': 14, 'num_leaves': 156, 'min_child_samples': 20, 'subsample': 0.8485692022239991, 'colsample_bytree': 0.9559385682531225}. Best is trial 0 with value: 0.9477336395127337.
[I 2025-11-21 10:19:08,680] Trial 1 finished with value: 0.951641296800429 and parameters: {'n_estimators': 407, 'learning_rate': 0.08525988675326297, 'max_depth': 6, 'num_leaves': 123, 'min_child_samples': 78, 'subsample': 0.9185208284361924, 'colsample_bytree': 0.8626801265660569}. Best is trial 1 with value: 0.951641296800429.
[I 2025-11-21 10:19:10,509] Trial 2 finished with value: 0.9515953587538706 and parameters: {'n_estimators': 554, 'learning_rate': 0.08027685621930405, 'max_depth': 7, 'num_leaves': 82, 'min_child_samples': 44, 'subsample': 0.7344388957389568, 'colsample_bytree': 0.7356835355300132}. Best is trial 1 with value: 0.951641296800429.
[


🎯 Best parameters: {'n_estimators': 486, 'learning_rate': 0.07505398239142168, 'max_depth': 6, 'num_leaves': 146, 'min_child_samples': 42, 'subsample': 0.7491432361359577, 'colsample_bytree': 0.7906485618457496}

=== CLASSIFICATION REPORT ===

              precision    recall  f1-score   support

           0       0.93      0.99      0.96      6332
           1       0.97      0.75      0.84      1772

    accuracy                           0.94      8104
   macro avg       0.95      0.87      0.90      8104
weighted avg       0.94      0.94      0.94      8104


=== CONFUSION MATRIX ===
[[6286   46]
 [ 447 1325]]

ROC-AUC Score: 0.9519179694240014

📌 Computing SHAP values...
📊 Generating SHAP summary plot...
SHAP plots saved in xai_outputs/

📌 Running LIME explanations...
✔ LIME explanation saved: lime_explanation_5778.html
✔ LIME explanation saved: lime_explanation_6179.html
✔ LIME explanation saved: lime_explanation_2984.html

📦 Model saved to xai_outputs/credit_risk_model.pkl

🎉

<Figure size 640x480 with 0 Axes>